In [ ]:
import pdfplumber
import docx
import re
import nltk
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

nltk.download('stopwords')

In [ ]:
def extract_text(file_path):
    if file_path.endswith(".pdf"):
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text += page.extract_text() or ""
        return text
    
    elif file_path.endswith(".docx"):
        doc = docx.Document(file_path)
        return " ".join([p.text for p in doc.paragraphs])
    
    else:
        raise ValueError("Unsupported file format")

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

In [ ]:
def tfidf_filter(resume, job_list, top_k=15):
    documents = [resume] + job_list
    
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents)
    
    resume_vec = tfidf_matrix[0]
    job_vecs = tfidf_matrix[1:]
    
    similarities = cosine_similarity(resume_vec, job_vecs)[0]
    
    # Get top-k indices
    top_indices = similarities.argsort()[::-1][:top_k]
    
    return top_indices, similarities

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
def bert_rerank(resume, job_list, top_indices):
    resume_emb = model.encode(resume)
    
    results = []
    
    for idx in top_indices:
        job_emb = model.encode(job_list[idx])
        
        sim = cosine_similarity([resume_emb], [job_emb])[0][0]
        score = sim * 100
        
        results.append((idx, score))
    
    # Sort by BERT score
    results.sort(key=lambda x: x[1], reverse=True)
    
    return results

In [ ]:
def job_matching_pipeline(resume_file, job_descriptions, top_k=15):
    
    # Step 1: Extract resume
    resume_text = extract_text(resume_file)
    
    # Step 2: Preprocess
    resume_clean = preprocess(resume_text)
    jobs_clean = [preprocess(j) for j in job_descriptions]
    
    # Step 3: TF-IDF Filtering
    top_indices, tfidf_scores = tfidf_filter(resume_clean, jobs_clean, top_k)
    
    # Step 4: BERT Ranking
    bert_results = bert_rerank(resume_clean, jobs_clean, top_indices)
    
    # Step 5: Prepare Output
    final_results = []
    
    for idx, score in bert_results:
        final_results.append({
            "job_id": idx,
            "match_score": round(score, 2),
            "job_description": job_descriptions[idx]
        })
    
    return final_results